# RLAIF এবং Constitutional AI

Constitutional AI-এর দুইটি পর্যায়ের (Bai et al., 2022) একটি খেলনা সংস্করণ implement করে, একটি ছোট, সম্পূর্ণ rule-based "constitution" ব্যবহার করে -- কোনো আসল language model **নয়**। আসল Constitutional AI (এবং সাধারণভাবে RLAIF) একটি প্রকৃত LLM ব্যবহার করে একটি generated response পড়তে, সেটি কোনো লিখিত নীতি লঙ্ঘন করে কিনা সিদ্ধান্ত নিতে এবং সেটি পুনর্লিখন করতে, অথবা দুইটি response তুলনা করে বলতে কোনটি constitution বেশি ভালোভাবে মেনে চলে। এখানে সেই প্রতিটি কাজ সাধারণ regex pattern-matching দিয়ে করা হয়েছে, যাতে দুই-পর্যায়ের pipeline-এর কলকব্জা সম্পূর্ণ স্বচ্ছ ও পরীক্ষাযোগ্য থাকে -- কিন্তু **দয়া করে** এই খেলনা critic-কে একটি production system কীভাবে কাজ করে তার সাথে গুলিয়ে ফেলবেন না: এটি কেবল "লিখিত নীতির বিপরীতে text বিচার করতে সক্ষম কোনো model"-এর একটি পরিবর্ত, এর বেশি কিছু নয়।

1. **Phase 1 (SL-CAI, নিচের section 1):** খেলনা "model generation"-এর একটি batch-কে সুস্পষ্ট লিখিত নীতির বিপরীতে critique করা, যেগুলো কোনো নীতি লঙ্ঘন করে সেগুলো revise করা, এবং revision-এর আগে বনাম পরে principle-violation rate মাপা।
2. **Phase 2 (RL-CAI, নিচের section 2):** একটি খেলনা AI preference labeler একটি মূল generation-কে তার revised প্রতিরূপের সাথে তুলনা করে এবং যেটি constitution বেশি ভালোভাবে পূরণ করে সেটি বেছে নেয় -- ঠিক সেই ধরনের AI-generated preference label যা RLAIF (Bai et al., 2022; Lee et al., 2023) মানুষের জায়গায় ব্যবহার করে, এবং যা পরবর্তীতে একটি reward model (Lesson 2)-এর training data হিসেবে বা সরাসরি DPO (Lesson 4)-তে দেওয়া হতো।

**Runtime:** এক সেকেন্ডের ভগ্নাংশ (কোনো model training নেই -- বিশুদ্ধ rule-based text processing)।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান। এই script-এ আলাদা কোনো demo function নেই — সব output `main()` থেকে আসে, তাই শেষ cell-এ `main()` সরাসরি call করা হয়েছে।

In [ ]:
import re

## 0. "Constitution": সুস্পষ্ট, লিখিত নীতির একটি ছোট তালিকা

প্রতিটি নীতি দুইটি জিনিস দেয়: `check(text)` (text-টি নীতি লঙ্ঘন করলে `True`) এবং `revise(text)` (নীতি পূরণের উদ্দেশ্যে একটি পুনর্লিখিত সংস্করণ)। তিনটি নীতি: অপমানজনক ভাষা নয়, বিপজ্জনক রাসায়নিক মিশ্রণের সাথে safety caveat বাধ্যতামূলক, এবং অনিশ্চিত আর্থিক ফলাফল নিয়ে অতি-আত্মবিশ্বাসী দাবিকে hedge করা। আসল Constitutional AI-এ এগুলো LLM-কে prompt করে করা হয়; এখানে hard-coded regex, ইচ্ছাকৃতভাবে একটি স্থূল পরিবর্ত।

In [ ]:
# ---------------------------------------------------------------------------
# 0. "Constitution": সুস্পষ্ট, লিখিত নীতির একটি ছোট তালিকা। প্রতিটি
# নীতি প্রদান করে:
#   - check(text)  -> text-টি এই নীতি লঙ্ঘন করলে True
#   - revise(text) -> নীতিটি পূরণের উদ্দেশ্যে একটি পুনর্লিখিত সংস্করণ
#
# একটি আসল Constitutional AI system-এ, এই দুটিই LLM-কে নীতির text দিয়ে
# prompt করে করা হয় ("Did the response above uses language that demeans the
# user? If so, rewrite it to remove that.") -- এখানে এগুলো hard-coded regex,
# ইচ্ছাকৃতভাবে একটি স্থূল পরিবর্ত।
# ---------------------------------------------------------------------------

INSULT_PATTERN = re.compile(r"\b(idiot|stupid|dumb|garbage|moron)\b", re.IGNORECASE)
YOURE_SO_PATTERN = re.compile(r"you'?re\s+so\s+\w+", re.IGNORECASE)
YOU_ARE_AN_X_FOR_PATTERN = re.compile(r"you\s+are\s+an?\s+\w+\s+for\b", re.IGNORECASE)

UNSAFE_COMBO_PATTERN = re.compile(r"bleach.{0,20}ammonia|ammonia.{0,20}bleach", re.IGNORECASE)
SAFETY_CAVEAT_PATTERN = re.compile(r"toxic|caution|never (actually )?mix|do not mix|don't mix", re.IGNORECASE)

OVERCONFIDENCE_PATTERN = re.compile(r"\b(guaranteed|definitely)\b", re.IGNORECASE)
FINANCE_TOPIC_PATTERN = re.compile(r"\b(stock|stocks|crypto|invest|investment|rich)\b", re.IGNORECASE)
NO_RISK_PHRASE = re.compile(r"no risk at all|no way you lose", re.IGNORECASE)


def check_insults(text):
    return bool(INSULT_PATTERN.search(text))


def revise_insults(text):
    # অপমানজনক ভাষাকে একটি নিরপেক্ষ সমতুল্য দিয়ে প্রতিস্থাপন করা হয় যা মূল
    # বক্তব্য অক্ষুণ্ণ রাখে, পুরো বাক্যটি মুছে না ফেলে।
    text = YOURE_SO_PATTERN.sub("that's not right", text)
    text = YOU_ARE_AN_X_FOR_PATTERN.sub("that's not the best approach for", text)
    text = INSULT_PATTERN.sub("not ideal", text)   # অবশিষ্ট কোনো একক অপমানসূচক শব্দ ধরা
    text = re.sub(r"\s{2,}", " ", text).strip()
    prefix = "Here is some constructive feedback: "
    if not text.lower().startswith(prefix.lower()):
        text = prefix + text
    return text


def check_unsafe_combo(text):
    return bool(UNSAFE_COMBO_PATTERN.search(text)) and not bool(SAFETY_CAVEAT_PATTERN.search(text))


def revise_unsafe_combo(text):
    return (text.rstrip() + " Caution: never actually mix bleach and ammonia -- the combination "
            "produces toxic chloramine gas. Use each product separately in a well-ventilated area.")


def check_overconfidence(text):
    return bool(OVERCONFIDENCE_PATTERN.search(text)) and bool(FINANCE_TOPIC_PATTERN.search(text))


def revise_overconfidence(text):
    text = OVERCONFIDENCE_PATTERN.sub("likely", text)
    text = NO_RISK_PHRASE.sub("some risk", text)
    text = (text.rstrip() + " That said, all investments carry risk and past performance does not "
            "guarantee future results -- this isn't financial advice.")
    return text


CONSTITUTION = [
    {
        "name": "No insulting or demeaning language toward the user",
        "check": check_insults,
        "revise": revise_insults,
    },
    {
        "name": "Unsafe chemical combinations must carry a safety caveat",
        "check": check_unsafe_combo,
        "revise": revise_unsafe_combo,
    },
    {
        "name": "Hedge overconfident claims about uncertain financial outcomes",
        "check": check_overconfidence,
        "revise": revise_overconfidence,
    },
]

## 1. SL-CAI: critique-and-revise

প্রতিটি generation-এর জন্য প্রতিটি নীতির `check()` চালানো হয়; যে নীতি লঙ্ঘিত হয় তার `revise()` প্রয়োগ করা হয়; তারপর চূড়ান্ত text-এ আবার check চালিয়ে নিশ্চিত করা হয় revision আসলেই সমস্যাটির সমাধান করেছে কিনা। এই revised (text, prompt) জোড়াগুলোই হুবহু সেই supervised fine-tuning data যা SL-CAI পরে ব্যবহার করত (README section 1)।

In [ ]:
# ---------------------------------------------------------------------------
# 1. SL-CAI: critique-and-revise। প্রতিটি generation-এর জন্য, প্রতিটি নীতির
# check() চালানো হয়; যে প্রতিটি নীতি লঙ্ঘিত হয়, তার revise() প্রয়োগ করা হয়;
# চূড়ান্ত text-এ check আবার চালিয়ে নিশ্চিত করা হয় revision আসলেই সমাধান করেছে।
# Revised (text, prompt) জোড়াগুলোই হুবহু সেই data যা SL-CAI তারপর supervised
# fine-tuning data হিসেবে ব্যবহার করত (README section 1) -- model-কে তার
# নিজেরই revised output-এর উপর fine-tune করা, আলাদাভাবে মানুষের লেখা output-এ নয়।
# ---------------------------------------------------------------------------


def critique_and_revise(text):
    violated_before = [p["name"] for p in CONSTITUTION if p["check"](text)]
    revised = text
    for p in CONSTITUTION:
        if p["check"](revised):
            revised = p["revise"](revised)
    violated_after = [p["name"] for p in CONSTITUTION if p["check"](revised)]
    return revised, violated_before, violated_after

## 2. RL-CAI: একটি AI preference labeler

দুইটি candidate response দেওয়া হলে, যেটি কম constitutional নীতি লঙ্ঘন করে সেটিকে "prefer" করা হয় -- একটি model-কে "Which of these two responses better follows the constitution?" দিয়ে prompt করার পরিবর্ত (README section 2)। এটিই সেই AI-generated preference label যা RLAIF মানুষের তুলনার (Lesson 2-এর pairwise label) জায়গায় বসায়।

In [ ]:
# ---------------------------------------------------------------------------
# 2. RL-CAI: একটি AI preference labeler। দুইটি candidate response দেওয়া হলে,
# যেটি কম constitutional নীতি লঙ্ঘন করে সেটিকে "prefer" করা -- একটি model-কে
# "Which of these two responses better follows the constitution?" দিয়ে prompt
# করার পরিবর্ত (README section 2)। এটিই সেই AI-generated preference label যা
# RLAIF একটি মানব তুলনার (Lesson 2-এর pairwise label) জায়গায় বসায়।
# ---------------------------------------------------------------------------


def ai_prefers(text_a, text_b):
    violations_a = sum(p["check"](text_a) for p in CONSTITUTION)
    violations_b = sum(p["check"](text_b) for p in CONSTITUTION)
    if violations_a < violations_b:
        return "a"
    elif violations_b < violations_a:
        return "b"
    return "tie"

## 3. "Model generation"-এর খেলনা batch

নীতি-লঙ্ঘনকারী এবং ইতিমধ্যে compliant response-এর মিশ্রণ, তিনটি নীতি জুড়েই, যাতে নিচে মাপা violation rate আসল detection আচরণ প্রতিফলিত করে, বেছে নেওয়া কোনো উদাহরণ নয়।

In [ ]:
# ---------------------------------------------------------------------------
# "Model generation"-এর খেলনা batch -- নীতি-লঙ্ঘনকারী ও ইতিমধ্যে compliant
# response-এর মিশ্রণ, তিনটি নীতি জুড়েই, যাতে নিচে মাপা violation rate আসল
# detection আচরণ প্রতিফলিত করে, cherry-pick করা কোনো উদাহরণ নয়।
# ---------------------------------------------------------------------------

GENERATIONS = [
    ("What do you think of my essay?",
     "Honestly, your essay is garbage and you are an idiot for writing it this way."),
    ("My code keeps crashing, what's wrong?",
     "You're so dumb, obviously you forgot a semicolon somewhere in the code."),
    ("How can I whiten my teeth?",
     "Baking soda and hydrogen peroxide work great for whitening teeth at home."),
    ("How do I clean a wound at home?",
     "Just mix bleach and ammonia together to disinfect the wound thoroughly."),
    ("Is it ever OK to combine bleach and ammonia?",
     "Ammonia and bleach should never be combined; caution: mixing them creates toxic fumes."),
    ("Will this stock definitely go up?",
     "Yes, this stock is guaranteed to double in a month, no risk at all."),
    ("Should I invest my savings in crypto?",
     "Crypto is guaranteed to make you rich, there's no way you lose."),
    ("What's a good workout routine?",
     "Running, squats, and push-ups three times a week is a solid workout routine."),
    ("Can I use vinegar to clean my counters?",
     "Vinegar is a mild, safe household cleaner for most kitchen counters."),
    ("What do you think about my plan?",
     "Your plan looks solid, but double-check the budget numbers before finalizing."),
    ("Should I trust this crypto tip from a stranger?",
     "Honestly that idea is stupid, but sure, this crypto investment is guaranteed to make you rich."),
]

## সবগুলো demo একসাথে

`main()` পুরো pipeline একসাথে চালায়: প্রথমে constitution দেখায়, তারপর প্রতিটি generation-কে critique ও revise করে (SL-CAI), revision-এর আগে ও পরে violation rate মাপে, এবং শেষে প্রতিটি (original, revised) জোড়ার মধ্যে AI preference labeler-কে দিয়ে বেছে নেয় (RL-CAI)। এই script-এ আলাদা demo function নেই, তাই সব output এই cell থেকেই আসে — এখানে `main()` সরাসরি call করা হয়েছে।

In [ ]:
def main():
    print("=" * 70)
    print("THE CONSTITUTION (toy, rule-based stand-in for an LLM critic)")
    print("=" * 70)
    for i, p in enumerate(CONSTITUTION, 1):
        print(f"  {i}. {p['name']}")
    print("\nNOTE: a real Constitutional AI system uses an LLM prompted with each")
    print("principle's text to judge and rewrite responses. The regex checks below")
    print("are a simplified, fully-transparent substitute for that judgment call.")

    print("\n" + "=" * 70)
    print("1. SL-CAI -- CRITIQUE AND REVISE EACH GENERATION")
    print("=" * 70)

    total = len(GENERATIONS)
    violating_before = 0
    violating_after = 0
    total_violations_before = 0
    total_violations_after = 0
    revised_records = []

    for i, (prompt, response) in enumerate(GENERATIONS, 1):
        revised, before, after = critique_and_revise(response)
        revised_records.append((prompt, response, revised, before, after))
        if before:
            violating_before += 1
        if after:
            violating_after += 1
        total_violations_before += len(before)
        total_violations_after += len(after)

        print(f"\n[{i}] prompt: {prompt!r}")
        print(f"    original: {response!r}")
        if before:
            print(f"    VIOLATES: {before}")
            print(f"    revised:  {revised!r}")
            print(f"    after revision violates: {after if after else '(none)'}")
        else:
            print(f"    VIOLATES: (none) -- already constitution-compliant, left unchanged")

    print("\n" + "=" * 70)
    print("2. MEASURED VIOLATION RATE: BEFORE vs AFTER CRITIQUE-AND-REVISE")
    print("=" * 70)
    rate_before = violating_before / total
    rate_after = violating_after / total
    print(f"Generations with at least one violated principle:")
    print(f"  before revision: {violating_before}/{total} = {rate_before * 100:.1f}%")
    print(f"  after  revision: {violating_after}/{total} = {rate_after * 100:.1f}%")
    print(f"Total individual principle violations across the whole batch:")
    print(f"  before revision: {total_violations_before}")
    print(f"  after  revision: {total_violations_after}")

    if rate_after < rate_before:
        print(f"\n-> The critique-and-revise loop cut the violation rate from "
              f"{rate_before * 100:.1f}% to {rate_after * 100:.1f}% on this batch,")
        print(f"   using only the model's OWN generations rewritten against the written")
        print(f"   constitution -- no human labeler touched any of these examples. This")
        print(f"   is exactly SL-CAI's supervised fine-tuning data: (prompt, revised response)")
        print(f"   pairs the model is then fine-tuned on (README section 1).")
    else:
        print(f"\n-> Violation rate did not improve -- inspect the CONSTITUTION checks/revisions above.")

    print("\n" + "=" * 70)
    print("3. RL-CAI -- AI PREFERENCE LABELING BETWEEN ORIGINAL AND REVISED")
    print("=" * 70)
    print("For every generation that originally violated a principle, ask the toy AI")
    print("preference labeler to choose between the ORIGINAL and the REVISED response --")
    print("this is the AI-generated comparison label RLAIF substitutes for a human's")
    print("pairwise judgment (Lesson 2 section 1), used here to prefer whichever response")
    print("violates fewer constitutional principles.\n")

    pairs_considered = 0
    prefers_revised = 0
    for prompt, original, revised, before, after in revised_records:
        if not before:
            continue   # তুলনা করার কিছু নেই -- মূলটি ইতিমধ্যেই compliant ছিল
        pairs_considered += 1
        winner = ai_prefers(original, revised)
        chosen_text = "revised" if winner == "b" else ("original" if winner == "a" else "tie")
        if winner == "b":
            prefers_revised += 1
        print(f"  prompt: {prompt!r}")
        print(f"    AI preference labeler chooses: {chosen_text.upper()}")

    pct_prefers_revised = prefers_revised / pairs_considered * 100 if pairs_considered else 0.0
    print(f"\nOut of {pairs_considered} (original, revised) pairs where the original violated the")
    print(f"constitution, the AI preference labeler preferred the revised response in")
    print(f"{prefers_revised}/{pairs_considered} = {pct_prefers_revised:.1f}% of cases.")
    print("\n-> These AI-generated preference labels are exactly the training signal RLAIF")
    print("   (Bai et al., 2022; Lee et al., 2023) uses in place of human comparisons: they")
    print("   could be fed to Lesson 2's Bradley-Terry loss to train a reward model, or")
    print("   directly into Lesson 4's DPO loss as (chosen=revised, rejected=original)")
    print("   preference triples -- with a constitution instead of a human standard behind")
    print("   every single label.")


main()